# BERT Yelp Sentiment Analysis

In [129]:
# Importing dependencies

# Data manipulation & visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Dataset handling
from datasets import Dataset
from imblearn.under_sampling import RandomUnderSampler

# Hugging Face
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# LoRA
from peft import LoraConfig, get_peft_model

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [112]:
# Load the cleaned CSV
reviews_df = pd.read_csv('../data/processed/yelp_reviews_cleaned.csv')
print(f"Loaded {len(reviews_df)} rows from CSV\n")
print(reviews_df.info())

Loaded 99966 rows from CSV

<class 'pandas.DataFrame'>
RangeIndex: 99966 entries, 0 to 99965
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   stars            99966 non-null  int64  
 1   text             99966 non-null  str    
 2   cleaned_text     99962 non-null  str    
 3   sentiment_score  99966 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 88.3 MB
None


In [113]:
# Map the sentiment labels (positive, negative and neutral) to corresponding star ratings in the DataFrame for training
reviews_df['label'] = reviews_df['stars'].replace({
    1: 0, # negative
    2: 0, 
    3: 1, # neutral
    4: 2,
    5: 2  # positive
})

In [139]:
# Check for class imbalance in the dataset
print(f"Distribution of labels in the dataset:\n{reviews_df['label'].value_counts().sort_index()}")

Distribution of labels in the dataset:
label
0    18902
1    11361
2    69703
Name: count, dtype: int64


In [140]:
# Define the data and target features
X = reviews_df[['cleaned_text']] # Convert to a DataFrame as RandomUnderSampler expects 2D features
y = reviews_df['label']

undersampler = RandomUnderSampler(sampling_strategy='not minority', random_state=42) # Resample all classes but the minority class, controlling randomization with a seed for reproducibility -- the magical 42 ("Answer to the Ultimate Question of Life")

X_res, y_res = undersampler.fit_resample(X, y)

print(f"Before undersampling: {Counter(y)}")
print(f"After undersampling: {Counter(y_res)}")

balanced_reviews_df = pd.DataFrame({
    'cleaned_text': X_res['cleaned_text'], # Convert from 2D DataFrame to 1D Series
    'label': y_res
})

print(f"\nDisplay the first few rows of the balanced dataset:\n{balanced_reviews_df.sort_index().head()}")

Before undersampling: Counter({2: 69703, 0: 18902, 1: 11361})
After undersampling: Counter({0: 11361, 1: 11361, 2: 11361})

Display the first few rows of the balanced dataset:
                                        cleaned_text  label
0  decide eat aware going take hours beginning en...      1
2  family diner buffet eclectic assortment large ...      1
5  long term frequent customer establishment went...      0
8  easter instead going lopez lake went los padre...      1
9  party hibachi waitress brought separate sushi ...      1


In [115]:
id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"} # A map from index to label
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2} # A map from label to index

In [116]:
# Add mapped label names column to the balanced DataFrame for readability
balanced_reviews_df['label_names'] = balanced_reviews_df['label'].replace(id2label)

# Convert the balanced DataFrame to a Hugging Face Dataset object
balanced_reviews_hf = Dataset.from_pandas(balanced_reviews_df)
print(balanced_reviews_hf[:20])

# Save the HuggingFace Dataset to disk
balanced_reviews_hf.save_to_disk('../data/processed/yelp_reviews_balanced_dataset')

{'cleaned_text': ['place mediocre restaurant serves canned beans fake mashed potatoes ranks low decor wornold wont going back', 'ive pizza probably dozen times im almost afraid post review seeing much positive feedback theyve getting money suppose good deal however pizza truly awful ive pizza least half pizza joints ridley area imperial bottom list yes theyre open late pretty much option id rather eat real sand ingredients taste cheap price youll get exactly pay stomach im completely drunk options', 'food location average best beef dip absolutely disgusting beef poor quality undercooked plain horrible wouldnt feed homeless dog', 'ive regular customer past months today decided last visit place service staff cooking staff not polite kept staring friends eating worth mentioning eat restaurant however order make us feel full quickly decided put quantity rice double size sushi left rice decided charge us not polite telling us youre ordering much quantities thats concluded put much rice girl

Saving the dataset (0/1 shards):   0%|          | 0/34083 [00:00<?, ? examples/s]

In [117]:
# Load the pre-trained BERT model for sequence classification, set the number of labels and the label mappings
model = AutoModelForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels=3, id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Creating a TextTokenizer Class



- `model_name="google-bert/bert-base-uncased"`: Define the default model parameter. If no model name is provided when creating an instance of this class, it will default to "google-bert/bert-base-uncased"




In [ ]:
class TextTokenizer:
    def __init__(self, model_name="google-bert/bert-base-uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name) # Load the tokenizer for the bert-base-uncased model
    
    def tokenize(self, text_batch, max_length=128): # Maximum sequence length set to default
        texts = [str(text) for text in text_batch["cleaned_text"]] # Make sure every input is a string with a list comprehension
        return self.tokenizer(texts,
                              max_length=max_length,
                              padding="max_length", # Ensures equal length
                              truncation=True # Truncates longer sequences to max_length
                              )
    
    # Define a method to convert the tokens back into human-readable format
    def decode(self, token_ids_batch, skip_special_tokens=True): # skip_special_tokens removes special tokens like [CLS], [SEP], and [PAD] from the decoded output for better readability
        return self.tokenizer.batch_decode(token_ids_batch, skip_special_tokens=skip_special_tokens)


# Create an instance (object) of the class
tokenizer_obj = TextTokenizer() # loads the tokenizer for "google-bert/bert-base-uncased" (default)

tokenized_dataset = balanced_reviews_hf.map(tokenizer_obj.tokenize, batched=True)

Map:   0%|          | 0/34083 [00:00<?, ? examples/s]

['place mediocre restaurant serves canned beans fake mashed potatoes ranks low decor wornold wont going back', 'ive pizza probably dozen times im almost afraid post review seeing much positive feedback theyve getting money suppose good deal however pizza truly awful ive pizza least half pizza joints ridley area imperial bottom list yes theyre open late pretty much option id rather eat real sand ingredients taste cheap price youll get exactly pay stomach im completely drunk options']

Keys in the tokenized dataset: ['cleaned_text', 'label', 'label_names', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']


In [135]:
# Display the keys of the tokenized dataset to verify the presence of 'input_ids', 'attention_mask', and 'label'
print(f"Keys in the tokenized dataset: {tokenized_dataset.column_names}\n")

# Display the tokenized sentences to verify the tokenization process
print(f"First 2 tokenized input_ids:\n{tokenized_dataset['input_ids'][:2]}\n")

# Display the first 2 decoded sequences from the tokenized dataset
print(f"First 2 decoded sequences:\n{tokenizer_obj.decode(tokenized_dataset['input_ids'][:2])}")

Keys in the tokenized dataset: ['cleaned_text', 'label', 'label_names', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']

First 2 tokenized input_ids:
[[101, 2173, 19960, 3695, 16748, 4825, 4240, 27141, 13435, 8275, 16137, 9072, 14629, 6938, 2659, 25545, 6247, 11614, 2180, 2102, 2183, 2067, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 4921, 2063, 10733, 2763, 6474, 2335, 10047, 2471, 4452, 2695, 3319, 3773, 2172, 3893, 12247, 2027, 3726, 2893, 2769, 6814, 2204, 3066, 2174, 10733, 5621, 9643, 4921, 2063, 10733, 2560, 2431, 10733, 17651, 20608, 2181, 4461, 3953, 2862, 2748, 2027, 2890, 2330, 2397, 3492, 2172, 5724, 8909, 2738, 4521, 2613, 5472, 12760, 5510, 10036, 3976, 2017, 3363, 2131, 35

## Splitting the Dataset

The `train_test_split` method automatically shuffles the data by default (`shuffle=True`).

__Parameters__:
- `test_size=0.2`: Define the size of the test set (`train_size` = 1 - `test_size`).
- `seed=42`: Set seed for reproducibility of the train/test split.

In [ ]:
# Split 80% train / 20% validation
dataset_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_set = dataset_split['train']
test_set = dataset_split['test']

print(f"Train size: {len(train_set)}, Validation size: {len(test_set)}")

Train size: 27266, Validation size: 6817


In [120]:
# Convert the Hugging Face Dataset Objects into PyTorch tensors for compatibility with BERT
train_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

## LoRA Configuration

- `r=16`: The rank, determines the number of trainable parameters.
- `lora_alpha=32`: The scaling factor set to be twice the rank, controls the magnitude of the updates during training.
- `target_modules=['query', 'value']`: Target only the query and value tensors as they have the strongest impact on model behaviour.
- `lora_dropout=0.1`: Regularization by adding small random stochastic noise to prevent overfitting and improve generalization.
- `bias='none'`: Do not train the bias terms in the model, only update the query and value tensors (as specified in `target_modules`).
- `task_type='SEQ_CLS'`: Define the task type as sequence classification for sentiment analysis.

In [130]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type='SEQ_CLS'
)

In [131]:
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

for name, param in model.named_parameters():
    print(name, param.requires_grad)

/Users/yevheniiakysel/Documents/Univesity Programming/Yelp-Reviews-Sentiment/project_env/lib/python3.13/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/yevheniiakysel/Documents/Univesity Programming/Yelp-Reviews-Sentiment/project_env/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 592,131 || all params: 110,076,678 || trainable%: 0.5379
bert.embeddings.word_embeddings.weight False
bert.embeddings.position_embeddings.weight False
bert.embeddings.token_type_embeddings.weight False
bert.embeddings.LayerNorm.weight False
bert.embeddings.LayerNorm.bias False
bert.encoder.layer.0.attention.self.query.base_layer.weight False
bert.encoder.layer.0.attention.self.query.base_layer.bias False
bert.encoder.layer.0.attention.self.query.lora_A.default.weight True
bert.encoder.layer.0.attention.self.query.lora_B.default.weight True
bert.encoder.layer.0.attention.self.key.weight False
bert.encoder.layer.0.attention.self.key.bias False
bert.encoder.layer.0.attention.self.value.base_layer.weight False
bert.encoder.layer.0.attention.self.value.base_layer.bias False
bert.encoder.layer.0.attention.self.value.lora_A.default.weight True
bert.encoder.layer.0.attention.self.value.lora_B.default.weight True
bert.encoder.layer.0.attention.output.dense.weight False
bert.en

## Training Configuration (`TrainingArguments`)

In this block, a `training_args` object is created to specify the training parameters for the Trainer.

- `output_dir="./lora_results"`: Automatically creates a "lora_results" directory to save model checkpoints and logs.
- `per_device_train_batch_size=8`: Defines the number of training sample batches processed on a GPU before the model's internal parameters are updated. Set to 8 to balance training speed and memory usage.
- `per_device_eval_batch_size=8`: Analogous for evaluation batch size, defines the number of evaluation sample batches processed on a GPU.
- `learning_rate=2e-4`: A relatively high learning rate for LoRA fine-tuning, as only a small number of parameters (low-rank matrices) are being updated, allowing for faster convergence.
- `num_train_epochs=3`:` A standard number of epochs for fine-tuning, means there will be three full passes over the dataset, allowing the model to update its weights at the end of each epoch. Set to <10 to avoid overfitting, can be adjusted based on validation performance.
- `eval_strategy="epoch"`: Evaluate the model at the end of each epoch to monitor performance.
- `save_strategy="epoch"`: Save the model updates at the and e=of each epoch.
- `logging_strategy="steps"`: Log training metrics every X steps (defined by logging_steps) instead of the end of every epoch for more frequent insights into the training process.
- `logging_steps=50`: Define the number of steps between logging events.
- `report_to='none'`: Disable integration with external logging platforms like WandB or TensorBoard, as local logging is sufficient for small projects.

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8, 
    learning_rate=2e-4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    report_to="none"
)

## Defining a function for computing evaluation metrics

- `eval_prediction`: A tuple storing the model's logits and the labels because the Trainer's compute_metrics callback expects a single argument.
- `logits`: The raw model output scores before applying softmax probability conversion.
- `labels`: The true class labels for the evaluation dataset.


- `np.argmax()`: Finds the index of the maximum value along the specified axis.
- `axis=-1`: 


__Accuracy__:
- `accuracy_score()`: A built in function from scikit-learn. Calculates the accuracy of the predictions throgh a ratio of correct predictions to total samples.
- Compares the predicted labels (`predictions`) with the true labels (`labels`).

__Precision, Recall, F-Score__:
- `precision_recall_fscore_support()`: A built-in function from scikit-learn, returns a tuple of `(precision, recall, f1, support)` for the specified average method.
- `average='weighted'`: Computes evaluation metrics for each class separately and returns a weighted average based on the number of true instances (support) in that class.
- `support(_)`: Support is the number of true positive instances for each class. Ignored with a `'_'` because it is not needed for the weighted average calculation.

In [ ]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction

    # Converting logits into predictions by taking the index of the highest logit value for each example with np.argmax
    predictions = np.argmax(logits, axis=-1) 

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted') # Each class's metrics are multiplied by its support and averaged over all classes

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=test_set,
    compute_metrics=compute_metrics
)

trainer.train()
evaluation_results = trainer.evaluate()
print(evaluation_results)

/Users/yevheniiakysel/Documents/Univesity Programming/Yelp-Reviews-Sentiment/project_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.455970,0.636268,0.750477,0.754991,0.750477,0.751244
2,0.467623,0.605210,0.761479,0.765910,0.761479,0.762785
3,0.510020,0.599663,0.765586,0.765808,0.765586,0.765553


/Users/yevheniiakysel/Documents/Univesity Programming/Yelp-Reviews-Sentiment/project_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yevheniiakysel/Documents/Univesity Programming/Yelp-Reviews-Sentiment/project_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yevheniiakysel/Documents/Univesity Programming/Yelp-Reviews-Sentiment/project_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: on_train_begin must be called before on_evaluate

In [ ]:
lora_model.save_pretrained("./lora_model")

merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained("./full_model")

In [ ]:
prediction_output = trainer.predict(test_set)
print(prediction_output.metrics) # Display the dictionary of evaluation metrics

In [ ]:
logits = prediction_output.predictions # Raw model predictions (logits) before softmax
labels = prediction_output.label_ids # True labels from the test set

predictions = np.argmax(logits, axis=1)

In [ ]:
confusion_matrix = confusion_matrix(labels, predictions)

## Creating a Seaborn Heatmap for Confusion Matrix Visualization

- `confusion_matrix`: The data parameter, a 2D array of counts
- `annot=True`: Displays the numerical values in each cell.
- `fmt='d'`: Formats the annotations as integers.
- `cmap='...'`: Sets the color scheme of the heatmap.
- `xticklabels=id2label.values()`: Custom labels for the x-axis.
- `yticklabels=id2label.values()`: Custom labels for the y-axis.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Pinks', xticklabels=id2label.values(), yticklabels=id2label.values())